In [2]:
# =========================
# AUTO-FILL FCD_id (fdcId) FROM FOOD NAMES
# =========================

import pandas as pd
import numpy as np
import requests
import time

# --- INPUTS ---
SHEET_CSV_URL = "https://docs.google.com/spreadsheets/d/e/2PACX-1vQwOKtoqUNXV0ag-KZsNvOureqjLz3H1BFwPoLuyEhdZi5_kvp2h-KPvc40VoziXBPjviI62Xl1oOdA/pub?output=csv"
USDA_API_KEY = "QgvQ1zpvzMyEORmbXPf5d5gTVzOONUCxzY1SR8a9"

FOOD_COL = "Food"     # from your screenshot
PRICE_COL = "Price"   # not used here, but kept
OUT_PATH = "foods_with_fcd_id.csv"

# --- USDA FDC SEARCH ---
SEARCH_URL = "https://api.nal.usda.gov/fdc/v1/foods/search"

# tweak this if you want different matching behavior
PREFERRED_DATA_TYPES = ["Foundation", "SR Legacy", "Survey (FNDDS)", "Branded"]  # priority order
PAGE_SIZE = 5
SLEEP_SEC = 0.15  # be nice to the API

def search_fdc(food_name: str, session: requests.Session):
    """
    Returns (fdcId, description, dataType) for best match, or (None, None, None)
    """
    if not food_name or not str(food_name).strip():
        return None, None, None

    payload = {
        "query": str(food_name),
        "pageSize": PAGE_SIZE,
        "pageNumber": 1,
        # you can uncomment to restrict:
        # "dataType": ["Foundation", "SR Legacy", "Survey (FNDDS)"]
    }

    r = session.post(SEARCH_URL, params={"api_key": USDA_API_KEY}, json=payload, timeout=30)
    r.raise_for_status()
    js = r.json()
    foods = js.get("foods", []) or []
    if not foods:
        return None, None, None

    # pick best match:
    # 1) first result with preferred datatype, else
    # 2) first result overall
    for dt in PREFERRED_DATA_TYPES:
        for f in foods:
            if f.get("dataType") == dt:
                return f.get("fdcId"), f.get("description"), f.get("dataType")

    f = foods[0]
    return f.get("fdcId"), f.get("description"), f.get("dataType")


# --- RUN ---
foods_df = pd.read_csv(SHEET_CSV_URL)

if FOOD_COL not in foods_df.columns:
    raise ValueError(f"Couldn't find '{FOOD_COL}' column. Columns are: {list(foods_df.columns)}")

# if FCD_id already exists, keep it; else create it
if "FCD_id" not in foods_df.columns:
    foods_df["FCD_id"] = np.nan

# add debug columns (optional but helpful)
if "FDC_match" not in foods_df.columns:
    foods_df["FDC_match"] = ""
if "FDC_type" not in foods_df.columns:
    foods_df["FDC_type"] = ""

session = requests.Session()
cache = {}  # food_name -> (fdcId, desc, dtype)

filled = 0
for i, name in enumerate(foods_df[FOOD_COL].astype(str).tolist()):
    if pd.notna(foods_df.loc[i, "FCD_id"]):
        continue  # already filled

    key = name.strip().lower()
    if key in cache:
        fdc_id, desc, dtype = cache[key]
    else:
        fdc_id, desc, dtype = search_fdc(name, session)
        cache[key] = (fdc_id, desc, dtype)
        time.sleep(SLEEP_SEC)

    if fdc_id is not None:
        foods_df.loc[i, "FCD_id"] = int(fdc_id)
        foods_df.loc[i, "FDC_match"] = desc or ""
        foods_df.loc[i, "FDC_type"] = dtype or ""
        filled += 1

print(f"✅ filled {filled} fdcIds")
missing = foods_df["FCD_id"].isna().sum()
print(f"still missing: {missing}")

foods_df.to_csv(OUT_PATH, index=False)
print("saved:", OUT_PATH)

✅ filled 176 fdcIds
still missing: 0
saved: foods_with_fcd_id.csv


In [30]:
import pandas as pd
import numpy as np
from scipy.optimize import linprog

# =========================
# 0) INPUT FILES (LOCAL CSVs)
# =========================
A_PATH = "matrix_A.csv"             # from nutrient notebook export
P_PATH = "price_vector_p.csv"       # from nutrient notebook export

# =========================
# 1) CONSTRAINT LINKS (pubhtml, no auth)
# =========================
CONSTRAINTS_HTML_BY_DISABILITY = {
    "No disability": "https://docs.google.com/spreadsheets/d/e/2PACX-1vRwJ2PiAGGKwIdHJYBnLP7bd0uT_qHJG1tqnAEQgSerlZOgxHigFfVZTxHbURUTCtkJJkPZTcDoiQ6L/pubhtml?gid=0&single=true",
    "Celiac": "https://docs.google.com/spreadsheets/d/e/2PACX-1vRwJ2PiAGGKwIdHJYBnLP7bd0uT_qHJG1tqnAEQgSerlZOgxHigFfVZTxHbURUTCtkJJkPZTcDoiQ6L/pubhtml?gid=761693646&single=true",
    "Diabetes": "https://docs.google.com/spreadsheets/d/e/2PACX-1vRwJ2PiAGGKwIdHJYBnLP7bd0uT_qHJG1tqnAEQgSerlZOgxHigFfVZTxHbURUTCtkJJkPZTcDoiQ6L/pubhtml?gid=1282759486&single=true",
    "Kidney disease": "https://docs.google.com/spreadsheets/d/e/2PACX-1vRwJ2PiAGGKwIdHJYBnLP7bd0uT_qHJG1tqnAEQgSerlZOgxHigFfVZTxHbURUTCtkJJkPZTcDoiQ6L/pubhtml?gid=1065244903&single=true",
}

# =========================
# 2) HELPERS
# =========================
def group_col(sex: str, age_group: str) -> str:
    # matches your sheet headers: Female_19_30, Male_31_50, etc.
    return f"{sex.strip().title()}_{age_group}"

def _coerce_numeric_series(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")

def load_constraints(disability: str, group: str) -> tuple[pd.Series, pd.Series]:
    if disability not in CONSTRAINTS_HTML_BY_DISABILITY:
        raise ValueError(f"Unknown disability '{disability}'. Choose from: {list(CONSTRAINTS_HTML_BY_DISABILITY)}")

    # pubhtml tables are readable via read_html
    df = pd.read_html(CONSTRAINTS_HTML_BY_DISABILITY[disability])[0]
    df.columns = [str(c).strip() for c in df.columns]

    # normalize key columns
    if "Nutrient" not in df.columns or "Constraint Type" not in df.columns:
        raise ValueError(f"Constraint sheet missing required columns. Found: {list(df.columns)}")

    df["Nutrient"] = df["Nutrient"].astype(str).str.strip()
    df["Constraint Type"] = df["Constraint Type"].astype(str).str.strip()

    if group not in df.columns:
        raise ValueError(f"Group column '{group}' not found. Available: {list(df.columns)}")

    vals = _coerce_numeric_series(df[group])

    # mins: RDA/AI, max: UL
    bmin = vals[df["Constraint Type"].isin(["RDA", "AI"])].copy()
    bmin.index = df.loc[bmin.index, "Nutrient"]

    bmax = vals[df["Constraint Type"].isin(["UL"])].copy()
    bmax.index = df.loc[bmax.index, "Nutrient"]

    # diabetes sheet uses "Carbohydrate max" as UL
    if "Carbohydrate max" in bmax.index:
        bmax.loc["Carbohydrate"] = float(bmax.loc["Carbohydrate max"])
        bmax = bmax.drop(index=["Carbohydrate max"])

    # drop NaNs
    bmin = bmin.dropna().astype(float)
    bmax = bmax.dropna().astype(float)

    return bmin, bmax

def load_A_and_p() -> tuple[pd.DataFrame, pd.Series]:
    # A exported from nutrient notebook is FOODS x NUTRIENTS (see your step 6) :contentReference[oaicite:1]{index=1}
    A_foods_by_nutrients = pd.read_csv(A_PATH, index_col=0)
    A_foods_by_nutrients.index = A_foods_by_nutrients.index.astype(str).str.strip()
    A_foods_by_nutrients.columns = A_foods_by_nutrients.columns.astype(str).str.strip()

    # transpose to NUTRIENTS x FOODS (this is what LP code expects)
    A = A_foods_by_nutrients.apply(pd.to_numeric, errors="coerce").fillna(0.0).T
    A.index = A.index.astype(str).str.strip()
    A.columns = A.columns.astype(str).str.strip()

    # p
    p_df = pd.read_csv(P_PATH, index_col=0)

    # handle whether p saved with header or without
    if p_df.shape[1] == 0:
        raise ValueError(f"{P_PATH} looks empty.")
    if p_df.shape[1] == 1:
        p = p_df.iloc[:, 0]
    else:
        # sometimes pandas writes a named column
        if "price_per_100g" in p_df.columns:
            p = p_df["price_per_100g"]
        else:
            # take first column
            p = p_df.iloc[:, 0]

    p.index = p.index.astype(str).str.strip()
    p = pd.to_numeric(p, errors="coerce")

    # align foods intersection
    foods = A.columns.intersection(p.index)
    A = A.loc[:, foods]
    p = p.loc[foods]

    # clean bad prices
    good = p.notna() & np.isfinite(p) & (p > 0)
    p = p[good]
    A = A.loc[:, p.index]

    if A.shape[1] == 0:
        raise ValueError("After aligning A and p, 0 foods left. Check food names in matrix_A.csv vs price_vector_p.csv.")

    return A, p

# =========================
# 3) SOLVER
# =========================
def solve_min_cost_diet(
    disability: str,
    sex: str,
    age_group: str,
    activity_energy_multiplier: float = 1.0,
    tol: float = 1e-6
) -> dict:

    group = group_col(sex, age_group)

    A, p = load_A_and_p()                 # A: nutrients x foods
    bmin, bmax = load_constraints(disability, group)

    # scale energy min only (if present)
    if "Energy" in bmin.index:
        bmin.loc["Energy"] = float(bmin.loc["Energy"]) * float(activity_energy_multiplier)

    # keep only nutrients that exist in A
    bmin = bmin[bmin.index.isin(A.index)]
    bmax = bmax[bmax.index.isin(A.index)]

    if len(bmin) == 0 and len(bmax) == 0:
        raise ValueError("No constraints matched any nutrient rows in A. Check nutrient naming between constraints + A.")

    # inequalities
    A_ub_list, b_ub_list = [], []

    # A x >= bmin  -> -A x <= -bmin
    if len(bmin) > 0:
        A_ub_list.append(-A.loc[bmin.index].to_numpy(dtype=float))
        b_ub_list.append(-bmin.to_numpy(dtype=float))

    # A x <= bmax
    if len(bmax) > 0:
        A_ub_list.append(A.loc[bmax.index].to_numpy(dtype=float))
        b_ub_list.append(bmax.to_numpy(dtype=float))

    A_ub = np.vstack(A_ub_list)
    b_ub = np.concatenate(b_ub_list)

    c = p.to_numpy(dtype=float).reshape(-1)

    # sanity check
    if A_ub.shape[1] != len(c):
        raise ValueError(f"Shape mismatch: A_ub has {A_ub.shape[1]} cols, c has {len(c)}")

    res = linprog(
        c=c,
        A_ub=A_ub,
        b_ub=b_ub,
        bounds=[(0, None)] * len(c),
        method="highs"
    )

    if not res.success:
        return {"success": False, "message": res.message}

    x = pd.Series(res.x, index=A.columns, name="units_of_100g")
    foods_used = x[x > tol].sort_values(ascending=False)

    outcomes = A.dot(x)
    check = pd.DataFrame({"Outcome": outcomes})
    check["MinReq"] = bmin.reindex(check.index)
    check["MaxAllowed"] = bmax.reindex(check.index)

    return {
        "success": True,
        "cost_per_day": float(res.fun),
        "foods_used": foods_used,
        "nutrient_check": check,
        "diet_vector": x,
    }

# =========================
# 4) RUN AN EXAMPLE
# =========================
r = solve_min_cost_diet("No disability", "Female", "19_30", 1.0)

print("success:", r["success"])
if r["success"]:
    print("cost_per_day:", r["cost_per_day"])
    print("\nTop foods used:")
    print(r["foods_used"].head(15))
    print("\nNutrient check (first 15 rows):")
    print(r["nutrient_check"].head(15))
else:
    print("message:", r["message"])

FileNotFoundError: [Errno 2] No such file or directory: 'matrix_A.csv'

In [32]:
import os, glob

print("cwd:", os.getcwd())

# look for the files anywhere under the current folder
print("matrix_A matches:", glob.glob("**/matrix_A.csv", recursive=True)[:20])
print("price_vector matches:", glob.glob("**/price_vector_p.csv", recursive=True)[:20])

cwd: /home/jovyan/EEP153_Materials/Project2
matrix_A matches: []
price_vector matches: []


In [22]:
import pandas as pd
import numpy as np
from scipy.optimize import linprog

# =========================
# 0) FILES YOU HAVE ON GIT
# =========================
A.to_csv('matrix_A.csv')
p.to_csv('price_vector_p.csv')
# If you don't have p_prices.csv but you have master_foods.csv instead, set:
# MASTER_PATH = "master_foods.csv"

# =========================
# 1) CONSTRAINT LINKS (pubhtml, no auth)
# =========================
CONSTRAINTS_HTML_BY_DISABILITY = {
    "No disability": "https://docs.google.com/spreadsheets/d/e/2PACX-1vRwJ2PiAGGKwIdHJYBnLP7bd0uT_qHJG1tqnAEQgSerlZOgxHigFfVZTxHbURUTCtkJJkPZTcDoiQ6L/pubhtml?gid=0&single=true",
    "Celiac": "https://docs.google.com/spreadsheets/d/e/2PACX-1vRwJ2PiAGGKwIdHJYBnLP7bd0uT_qHJG1tqnAEQgSerlZOgxHigFfVZTxHbURUTCtkJJkPZTcDoiQ6L/pubhtml?gid=761693646&single=true",
    "Diabetes": "https://docs.google.com/spreadsheets/d/e/2PACX-1vRwJ2PiAGGKwIdHJYBnLP7bd0uT_qHJG1tqnAEQgSerlZOgxHigFfVZTxHbURUTCtkJJkPZTcDoiQ6L/pubhtml?gid=1282759486&single=true",
    "Kidney disease": "https://docs.google.com/spreadsheets/d/e/2PACX-1vRwJ2PiAGGKwIdHJYBnLP7bd0uT_qHJG1tqnAEQgSerlZOgxHigFfVZTxHbURUTCtkJJkPZTcDoiQ6L/pubhtml?gid=1065244903&single=true",
}

# =========================
# 2) HELPERS
# =========================
def group_col(sex: str, age_group: str) -> str:
    # expects your constraint sheet columns like Female_19_30, Male_31_50, etc.
    return f"{sex.strip().title()}_{age_group}"

def load_constraints(disability: str, group: str) -> tuple[pd.Series, pd.Series]:
    df = pd.read_html(CONSTRAINTS_HTML_BY_DISABILITY[disability])[0]
    df.columns = [str(c).strip() for c in df.columns]
    df["Nutrient"] = df["Nutrient"].astype(str).str.strip()
    df["Constraint Type"] = df["Constraint Type"].astype(str).str.strip()

    need = {"Nutrient", "Constraint Type", group}
    missing = need - set(df.columns)
    if missing:
        raise ValueError(f"Constraints table missing {missing}. Found: {list(df.columns)}")

    bmin = (df[df["Constraint Type"].isin(["RDA", "AI"])].set_index("Nutrient")[group]).astype(float)
    bmax = (df[df["Constraint Type"].isin(["UL"])].set_index("Nutrient")[group]).astype(float)

    # diabetes special: some sheets use "Carbohydrate max"
    if "Carbohydrate max" in bmax.index:
        bmax.loc["Carbohydrate"] = float(bmax.loc["Carbohydrate max"])
        bmax = bmax.drop(index=["Carbohydrate max"])

    return bmin, bmax

def load_A_and_p() -> tuple[pd.DataFrame, pd.Series]:
    # A: nutrients x foods
    A = pd.read_csv(A_PATH, index_col=0)
    A.index = A.index.astype(str).str.strip()
    A.columns = A.columns.astype(str).str.strip()

    # p: foods -> price
    p_df = pd.read_csv(P_PATH, index_col=0)
    if p_df.shape[1] == 1:
        p = p_df.iloc[:, 0]
    else:
        # if multiple cols, try common name
        if "price_per_100g" in p_df.columns:
            p = p_df["price_per_100g"]
        else:
            raise ValueError(f"{P_PATH} has multiple columns; expected 1. Columns: {list(p_df.columns)}")

    p.index = p.index.astype(str).str.strip()

    # align to intersection
    foods = A.columns.intersection(p.index)
    A = A.loc[:, foods]
    p = p.loc[foods]

    if A.shape[1] == 0:
        raise ValueError("After aligning A and p, there are 0 foods left. Check column/index names.")
    return A, p

# =========================
# 3) SOLVER
# =========================
def solve_min_cost_diet(
    disability: str,
    sex: str,
    age_group: str,
    activity_energy_multiplier: float = 1.0,
    tol: float = 1e-6
) -> dict:

    group = group_col(sex, age_group)

    A, p = load_A_and_p()                 # A: nutrients x foods, p: foods
    bmin, bmax = load_constraints(disability, group)

    # scale energy min for activity if present
    if "Energy" in bmin.index:
        bmin.loc["Energy"] = float(bmin.loc["Energy"]) * float(activity_energy_multiplier)

    # keep only nutrients that exist in A
    bmin = bmin[bmin.index.isin(A.index)]
    bmax = bmax[bmax.index.isin(A.index)]

    # Build inequalities
    A_ub_list, b_ub_list = [], []

    # A x >= bmin  -> -A x <= -bmin
    if len(bmin) > 0:
        A_ub_list.append(-A.loc[bmin.index].to_numpy(dtype=float))
        b_ub_list.append(-bmin.to_numpy(dtype=float))

    # A x <= bmax
    if len(bmax) > 0:
        A_ub_list.append(A.loc[bmax.index].to_numpy(dtype=float))
        b_ub_list.append(bmax.to_numpy(dtype=float))

    if not A_ub_list:
        raise ValueError("No matching constraints found in A rows. Check nutrient names.")

    A_ub = np.vstack(A_ub_list)
    b_ub = np.concatenate(b_ub_list)

    c = p.to_numpy(dtype=float).reshape(-1)

    # sanity check
    if A_ub.shape[1] != len(c):
        raise ValueError(f"Shape mismatch: A_ub has {A_ub.shape[1]} cols, c has {len(c)}")

    res = linprog(
        c=c,
        A_ub=A_ub,
        b_ub=b_ub,
        bounds=[(0, None)] * len(c),
        method="highs"
    )

    if not res.success:
        return {"success": False, "message": res.message}

    x = pd.Series(res.x, index=A.columns, name="units_of_100g")
    foods_used = x[x > tol].sort_values(ascending=False)

    outcomes = A.dot(x)
    check = pd.DataFrame({"Outcome": outcomes})
    check["MinReq"] = bmin.reindex(check.index)
    check["MaxAllowed"] = bmax.reindex(check.index)

    return {
        "success": True,
        "cost_per_day": float(res.fun),
        "foods_used": foods_used,
        "nutrient_check": check,
        "diet_vector": x,
    }

# =========================
# 4) RUN AN EXAMPLE
# =========================
r = solve_min_cost_diet("No disability", "Female", "19_30", 1.0)

print("success:", r["success"])
if r["success"]:
    print("cost_per_day:", r["cost_per_day"])
    print("\nTop foods used:")
    print(r["foods_used"].head(15))
    print("\nNutrient check (top 15 rows):")
    print(r["nutrient_check"].head(15))
else:
    print("message:", r["message"])

NameError: name 'A' is not defined

In [33]:
import os
print("Solver cwd:", os.getcwd())
print("Files here:", os.listdir())

Solver cwd: /home/jovyan/EEP153_Materials/Project2
Files here: ['george_stigler-1c2ecd00695f.json.gpg', 'goals.pdf', 'stigler45.pdf', 'Food prices.ipynb', 'minimum_cost_diet_slides.pdf', 'james_lind-f8fb7d72a9d3.json.gpg', 'harriette-chick.json.gpg', 'unitsrc', 'foods_with_fcd_id.csv', 'requirements.txt', 'wilbur_atwater-4a4660352992.json.gpg', 'dantzig90.pdf', 'justus_von_liebig-069d38b622a8.json.gpg', 'diet_maximums.csv', 'nutrient_mapping.ipynb', 'diet_minimums.csv', 'diet_problem2.ipynb', 'min_cost_diet_problem.ipynb', 'master_foods.csv', 'fndds_diet_problem.ipynb', '.ipynb_checkpoints', 'casimir_funk-608948767d01.json.gpg', 'students.json.gpg', 'margaret_reid-75efa1586b05.json.gpg', 'fdc_nutrients_cache.json', 'diet_problem.ipynb', 'Data']
